# 03 — Feature Engineering (Review 2)

**Owner: Abhishek (data engineering)** — builds on the EDA/preprocessing notebook.

**Why this notebook exists.** The Review 1 data was forward-filled and then median-imputed. That made every row
complete, but it also erased *when* each test was last measured (informative in itself: frequent lab draws signal clinical concern),
and it gave the models no features describing *how vitals are trending*. This notebook rebuilds the features from
`sepsis_combined_raw.csv` (so nothing is lost) using **the exact same train/test patient split as Review 1**,
so all Review 1 vs Review 2 comparisons are fair.

**Rules followed (all features look backwards in time only):** forward-fill within a patient, trailing rolling windows,
lag differences, cumulative max/min. A check further down proves there is no look-ahead.

**Output:** `review2_train.joblib`, `review2_test.joblib` (in `data/processed/`).

In [1]:
# CELL 1 — Imports and project paths (everything is relative to the repository root)
import os
import joblib
import numpy as np
import pandas as pd
from pathlib import Path

def find_project_root():
    here = Path.cwd()
    for folder in [here, here.parent]:
        if (folder / "notebooks").is_dir():
            return folder
    raise FileNotFoundError("Open this notebook from the repository root or from the notebooks/ folder.")

PROJECT_ROOT = find_project_root()

PROC_DIR = PROJECT_ROOT / "data" / "processed"
PROC_DIR.mkdir(parents=True, exist_ok=True)

RAW_CSV = PROC_DIR / "sepsis_combined_raw.csv"   # written by 01_EDA_and_Preprocessing (CELL 7)
MODEL_DATA = PROC_DIR / "model_data.joblib"      # written by 01_EDA_and_Preprocessing (CELL 32)

for path in [RAW_CSV, MODEL_DATA]:
    if not path.exists():
        raise FileNotFoundError(f"Missing {path.relative_to(PROJECT_ROOT)} - run 01_EDA_and_Preprocessing.ipynb first.")

OUT_DIR = PROC_DIR

print("Raw CSV      :", RAW_CSV.relative_to(PROJECT_ROOT))
print("Model data   :", MODEL_DATA.relative_to(PROJECT_ROOT))
print("Output folder:", OUT_DIR.relative_to(PROJECT_ROOT))

Raw CSV      : data\processed\sepsis_combined_raw.csv
Model data   : data\processed\model_data.joblib
Output folder: data\processed


In [2]:
# CELL 2 — Column groups
VITALS = ["HR", "O2Sat", "Temp", "SBP", "MAP", "DBP", "Resp", "EtCO2"]

LABS = [
    "BaseExcess", "HCO3", "FiO2", "pH", "PaCO2", "SaO2", "AST", "BUN", "Alkalinephos",
    "Calcium", "Chloride", "Creatinine", "Bilirubin_direct", "Glucose", "Lactate",
    "Magnesium", "Phosphate", "Potassium", "Bilirubin_total", "TroponinI",
    "Hct", "Hgb", "PTT", "WBC", "Fibrinogen", "Platelets",
]

DEMOGRAPHICS = ["Age", "Gender", "Unit1", "Unit2", "HospAdmTime", "ICULOS"]

TIME_SERIES_COLS = VITALS + LABS            # 34 measured columns
RAW_COLS = TIME_SERIES_COLS + DEMOGRAPHICS  # the 40 Review 1 features
CORE_VITALS = ["HR", "O2Sat", "Temp", "SBP", "MAP", "DBP", "Resp"]  # dense enough for rolling stats

print("Measured columns:", len(TIME_SERIES_COLS))
print("Review 1 raw features:", len(RAW_COLS))

Measured columns: 34
Review 1 raw features: 40


In [3]:
# CELL 3 — Load raw combined data and the Review 1 patient split
float_cols = TIME_SERIES_COLS + ["Age", "Unit1", "Unit2", "HospAdmTime"]

raw = pd.read_csv(RAW_CSV, dtype={c: "float32" for c in float_cols})

raw["Patient_Key"] = raw["Hospital_Set"].astype(str) + "_" + raw["Patient_ID"].astype(str)

# Chronological order inside every patient (same ordering as Review 1)
raw = raw.sort_values(["Patient_Key", "ICULOS"]).reset_index(drop=True)

saved = joblib.load(MODEL_DATA)
train_keys = set(saved["train_patient_keys"])
test_keys = set(saved["test_patient_keys"])
del saved

print("Raw shape          :", raw.shape)
print("Unique patients    :", raw["Patient_Key"].nunique())
print("Train patients     :", len(train_keys))
print("Test patients      :", len(test_keys))
print("Train/test overlap :", len(train_keys & test_keys))

Raw shape          : (1163798, 44)
Unique patients    : 30338
Train patients     : 24270
Test patients      : 6068
Train/test overlap : 0


In [4]:
# CELL 4 — Feature engineering function
# The data must be sorted by Patient_Key and ICULOS before calling this.
# Every feature uses only the current hour and PAST hours of the same patient.

def build_features(df):
    key = df["Patient_Key"]
    new = {}

    def add(name, series):
        new[name] = series.astype("float32")

    # 1. Hours since each vital / lab was last actually measured.
    #    (Computed BEFORE forward-filling, otherwise the information is lost.)
    for col in TIME_SERIES_COLS:
        last_seen = df["ICULOS"].where(df[col].notna())
        last_seen = last_seen.groupby(key).ffill()
        add(col + "_hrs_since", df["ICULOS"] - last_seen)

    # 2. Number of lab tests drawn this hour and so far in the stay
    labs_now = df[LABS].notna().sum(axis=1)
    add("labs_measured_now", labs_now)
    add("labs_measured_cum", labs_now.groupby(key).cumsum())

    # 3. Forward-filled vitals and labs (NaN is kept until a value is first seen)
    filled = df[TIME_SERIES_COLS].groupby(key).ffill()
    for col in TIME_SERIES_COLS:
        add(col, filled[col])

    # 4. Demographics and ICU time (unchanged)
    for col in DEMOGRAPHICS:
        add(col, df[col])

    # 5. Rolling statistics over the last 6 hours for the core vitals
    vitals_filled = filled[CORE_VITALS]
    roll_1 = vitals_filled.groupby(key).rolling(window=6, min_periods=1)
    roll_2 = vitals_filled.groupby(key).rolling(window=6, min_periods=2)

    rolling_results = {
        "mean6": roll_1.mean(),
        "min6": roll_1.min(),
        "max6": roll_1.max(),
        "std6": roll_2.std(),
    }
    for stat_name, stat_df in rolling_results.items():
        stat_df = stat_df.reset_index(level=0, drop=True).sort_index()
        for col in CORE_VITALS:
            add(col + "_" + stat_name, stat_df[col])

    # 6. Trend: change over the last 3 hours
    change_3h = vitals_filled.groupby(key).diff(3)
    for col in CORE_VITALS:
        add(col + "_chg3", change_3h[col])

    # 7. Clinical scores (domain knowledge as features)
    hr = filled["HR"]
    sbp = filled["SBP"]

    add("shock_index", hr / sbp)
    add("pulse_pressure", sbp - filled["DBP"])
    add("bun_creat_ratio", filled["BUN"] / filled["Creatinine"])

    sirs_temp = (filled["Temp"] > 38) | (filled["Temp"] < 36)
    sirs_hr = hr > 90
    sirs_resp = (filled["Resp"] > 20) | (filled["PaCO2"] < 32)
    sirs_wbc = (filled["WBC"] > 12) | (filled["WBC"] < 4)
    add("sirs_count", sirs_temp.astype(int) + sirs_hr.astype(int) + sirs_resp.astype(int) + sirs_wbc.astype(int))

    add("qsofa_partial", (filled["Resp"] >= 22).astype(int) + (sbp <= 100).astype(int))
    add("map_below_65", (filled["MAP"] < 65).astype(int))
    add("lactate_above_2", (filled["Lactate"] > 2).astype(int))

    # 8. Worst value so far in the ICU stay
    for col in ["HR", "Resp", "Temp"]:
        add(col + "_max_so_far", filled[col].groupby(key).cummax())
    for col in ["SBP", "MAP", "O2Sat"]:
        add(col + "_min_so_far", filled[col].groupby(key).cummin())

    feats = pd.DataFrame(new, index=df.index)
    feats = feats.replace([np.inf, -np.inf], np.nan)
    return feats

print("build_features() defined.")

build_features() defined.


In [5]:
# CELL 5 — Leakage check: features must not depend on FUTURE hours
# Recompute features after cutting every patient in half. If the first-half
# features are identical, then no feature can be using future information.

def leakage_check(df, n_patients=300, seed=0):
    rng = np.random.default_rng(seed)
    all_keys = df["Patient_Key"].unique()
    chosen = rng.choice(all_keys, size=min(n_patients, len(all_keys)), replace=False)

    sub = df[df["Patient_Key"].isin(chosen)].reset_index(drop=True)
    full_feats = build_features(sub)

    hours_in_stay = sub.groupby("Patient_Key")["ICULOS"].transform("size")
    position = sub.groupby("Patient_Key").cumcount()
    keep = (position < hours_in_stay // 2).values

    cut = sub[keep].reset_index(drop=True)
    cut_feats = build_features(cut)
    full_first_half = full_feats[keep].reset_index(drop=True)

    bad_cols = []
    for col in full_feats.columns:
        same = np.allclose(full_first_half[col], cut_feats[col], equal_nan=True, atol=1e-4)
        if not same:
            bad_cols.append(col)
    return bad_cols

bad_cols = leakage_check(raw)

if len(bad_cols) == 0:
    print("PASSED: all features are identical when future hours are removed (no look-ahead).")
else:
    print("FAILED: these features use future information:", bad_cols)

PASSED: all features are identical when future hours are removed (no look-ahead).


In [6]:
# CELL 6 — Build features for the full dataset
features = build_features(raw)

print("Feature matrix shape:", features.shape)
print("Number of features  :", features.shape[1])
print("Memory (GB)         :", round(features.memory_usage(deep=True).sum() / 1e9, 2))

Feature matrix shape: (1163798, 124)
Number of features  : 124
Memory (GB)         : 0.58


In [7]:
# CELL 7 — Split with the SAME patients as Review 1 and check the sizes
y = raw["SepsisLabel"].astype("int8")
meta = raw[["Patient_Key", "ICULOS"]]

is_train = raw["Patient_Key"].isin(train_keys).values
is_test = raw["Patient_Key"].isin(test_keys).values

assert (is_train | is_test).all(), "Some rows belong to neither split"
assert not (is_train & is_test).any(), "Some rows belong to both splits"

X_train = features[is_train].reset_index(drop=True)
X_test = features[is_test].reset_index(drop=True)
y_train = y[is_train].reset_index(drop=True)
y_test = y[is_test].reset_index(drop=True)
meta_train = meta[is_train].reset_index(drop=True)
meta_test = meta[is_test].reset_index(drop=True)

print("X_train:", X_train.shape, "| Review 1 had (1240596, 40)")
print("X_test :", X_test.shape, "| Review 1 had (311614, 40)")
print("Positive rate train: %.4f" % y_train.mean())
print("Positive rate test : %.4f" % y_test.mean())

X_train: (931285, 124) | Review 1 had (1240596, 40)
X_test : (232513, 124) | Review 1 had (311614, 40)
Positive rate train: 0.0169
Positive rate test : 0.0168


In [8]:
# CELL 8 — Feature families and missingness (for the report)
def family(name):
    if name.endswith("_hrs_since"):
        return "Hours since last measurement"
    if name.endswith(("_mean6", "_std6", "_min6", "_max6")):
        return "Rolling 6-hour statistics"
    if name.endswith("_chg3"):
        return "3-hour change (trend)"
    if name.endswith(("_max_so_far", "_min_so_far")):
        return "Worst value so far"
    if name in ["shock_index", "pulse_pressure", "bun_creat_ratio", "sirs_count",
                "qsofa_partial", "map_below_65", "lactate_above_2"]:
        return "Clinical scores"
    if name in ["labs_measured_now", "labs_measured_cum"]:
        return "Lab-draw counts"
    return "Review 1 raw features (forward-filled)"

family_table = (
    pd.Series({c: family(c) for c in X_train.columns})
    .value_counts()
    .rename_axis("Feature family")
    .reset_index(name="Count")
)
display(family_table)

missing_pct = (X_train.isna().mean() * 100).sort_values(ascending=False)
print("\nTop 10 features by remaining missing % (trees handle NaN natively):")
display(missing_pct.head(10).round(2).to_frame("Missing_%"))

,Feature family,Count
0,Review 1 raw features (forward-filled),40
1,Hours since last measurement,34
2,Rolling 6-hour statistics,28
3,3-hour change (trend),7
4,Clinical scores,7
5,Worst value so far,6
6,Lab-draw counts,2



Top 10 features by remaining missing % (trees handle NaN natively):


,Missing_%
Bilirubin_direct_hrs_since,94.49
Bilirubin_direct,94.49
Fibrinogen_hrs_since,89.54
Fibrinogen,89.54
EtCO2,88.75
EtCO2_hrs_since,88.75
TroponinI,80.42
TroponinI_hrs_since,80.42
BaseExcess,77.57
BaseExcess_hrs_since,77.57


In [9]:
# CELL 9 — Save for the modelling notebooks
# Train and test are saved separately so the feature-selection notebook
# never even loads the test set.

joblib.dump(
    {"X": X_train, "y": y_train, "meta": meta_train,
     "feature_names": list(X_train.columns), "raw_columns": RAW_COLS},
    os.path.join(OUT_DIR, "review2_train.joblib"),
    compress=3,
)

joblib.dump(
    {"X": X_test, "y": y_test, "meta": meta_test,
     "feature_names": list(X_test.columns), "raw_columns": RAW_COLS},
    os.path.join(OUT_DIR, "review2_test.joblib"),
    compress=3,
)

print("Saved:")
print(" ", os.path.join(OUT_DIR, "review2_train.joblib"))
print(" ", os.path.join(OUT_DIR, "review2_test.joblib"))

Saved:
  d:\SepsiChronos HDA\data\processed\review2_train.joblib
  d:\SepsiChronos HDA\data\processed\review2_test.joblib
